In [1]:
import polars as pl

trades_raw = pl.read_parquet("../parquets/A_hkex_2800_trades.parquet")

trades = trades_raw.unique()

pl.DataFrame({
    "stage": ["raw", "deduped"],
    "rows": [trades_raw.height, trades.height],
    "columns": [trades_raw.width, trades.width],
})

trades = trades_raw.unique()

trades_raw.shape, trades.shape

((9666, 9), (9666, 9))

In [2]:
trades.group_by("trade_id").len().filter(pl.col("len") > 1)

trade_id,len
str,u32


In [3]:
trades = trades.with_columns(
    trade_id_num=pl.col("trade_id").cast(pl.Int64, strict=False)
)
trades.filter(pl.col("trade_id_num").is_null())

instrument,ingress_ts,transaction_ts,publish_ts,trade_id,price,qty,side,other_data,trade_id_num
str,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,str,str,i64


In [4]:
trades.sort("ingress_ts").with_columns(
    trade_id_diff=pl.col("trade_id_num").diff()
).filter(
    pl.col("trade_id_diff") < 0
)

instrument,ingress_ts,transaction_ts,publish_ts,trade_id,price,qty,side,other_data,trade_id_num,trade_id_diff
str,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,str,str,i64,i64
"""S|2800-HKD:SPOT""",2026-08-13 01:20:03.835334 UTC,2026-08-13 01:20:03.782040 UTC,2026-08-13 01:20:03.782040 UTC,"""10""",25.8,500.0,"""Buy""",null,10,-7
"""S|2800-HKD:SPOT""",2026-08-13 01:20:03.835334 UTC,2026-08-13 01:20:03.782040 UTC,2026-08-13 01:20:03.782040 UTC,"""8""",25.8,500.0,"""Buy""",null,8,-12
"""S|2800-HKD:SPOT""",2026-08-13 01:20:03.835334 UTC,2026-08-13 01:20:03.781810 UTC,2026-08-13 01:20:03.781810 UTC,"""2""",25.8,2000.0,"""Buy""",null,2,-17
"""S|2800-HKD:SPOT""",2026-08-13 01:20:03.835334 UTC,2026-08-13 01:20:03.782040 UTC,2026-08-13 01:20:03.782040 UTC,"""11""",25.8,2000.0,"""Buy""",null,11,-2
"""S|2800-HKD:SPOT""",2026-08-13 01:20:03.835334 UTC,2026-08-13 01:20:03.781810 UTC,2026-08-13 01:20:03.781810 UTC,"""5""",25.8,500.0,"""Buy""",null,5,-6
…,…,…,…,…,…,…,…,…,…,…
"""S|2800-HKD:SPOT""",2026-08-13 08:08:12.739081 UTC,2026-08-13 08:08:12.707420 UTC,2026-08-13 08:08:12.707420 UTC,"""9647""",25.88,8000.0,"""Buy""",null,9647,-8
"""S|2800-HKD:SPOT""",2026-08-13 08:08:12.739081 UTC,2026-08-13 08:08:12.707420 UTC,2026-08-13 08:08:12.707420 UTC,"""9642""",25.88,500.0,"""Buy""",null,9642,-17
"""S|2800-HKD:SPOT""",2026-08-13 08:08:12.739081 UTC,2026-08-13 08:08:12.706680 UTC,2026-08-13 08:08:12.706680 UTC,"""9638""",25.88,50000.0,"""Buy""",null,9638,-23


In [5]:
for col in ["ingress_ts", "transaction_ts", "publish_ts"]:
    dupes = trades.group_by(col).len().filter(pl.col("len") > 1)
    print(col, "duplicate timestamp values:", dupes.height, "rows involved:", dupes["len"].sum())

ingress_ts duplicate timestamp values: 1527 rows involved: 5960
transaction_ts duplicate timestamp values: 1101 rows involved: 4796
publish_ts duplicate timestamp values: 1450 rows involved: 4462


In [8]:
trades.with_columns(
    venue_delay_us=(pl.col("publish_ts") - pl.col("transaction_ts")).dt.total_microseconds()
).select(
    pl.col("venue_delay_us").min().alias("min"),
    pl.col("venue_delay_us").quantile(0.5).alias("median"),
    pl.col("venue_delay_us").quantile(0.95).alias("p95"),
    pl.col("venue_delay_us").quantile(0.99).alias("p99"),
    pl.col("venue_delay_us").max().alias("max"),
)

min,median,p95,p99,max
i64,f64,f64,f64,i64
0,0.0,0.0,0.0,0


In [9]:
trades.filter(pl.col("publish_ts") < pl.col("transaction_ts"))

instrument,ingress_ts,transaction_ts,publish_ts,trade_id,price,qty,side,other_data,trade_id_num
str,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,str,str,i64


In [10]:
trades.select(
    pl.len().alias("rows"),
    pl.col("transaction_ts").null_count().alias("missing_transaction_ts"),
    pl.col("publish_ts").null_count().alias("missing_publish_ts"),
    ((pl.col("transaction_ts") == pl.col("publish_ts")) | pl.col("transaction_ts").is_null()).sum().alias("transaction_equals_publish_or_null"),
)

rows,missing_transaction_ts,missing_publish_ts,transaction_equals_publish_or_null
u32,u32,u32,u32
9666,1513,0,9666


In [11]:
trades = trades.sort(["publish_ts"])

In [13]:
trades.group_by(
    ["ingress_ts", "publish_ts", "price", "qty", "side"]
).len().filter(
    pl.col("len") > 1
).sort("len", descending=True)

ingress_ts,publish_ts,price,qty,side,len
"datetime[μs, UTC]","datetime[μs, UTC]",f64,f64,str,u32
2026-08-13 01:20:03.835334 UTC,2026-08-13 01:20:03.782040 UTC,25.8,500.0,"""Buy""",8
2026-08-13 06:30:35.133530 UTC,2026-08-13 06:30:35.099100 UTC,25.88,500.0,"""Buy""",7
2026-08-13 07:07:06.834784 UTC,2026-08-13 07:07:06.803320 UTC,25.84,500.0,"""Buy""",6
2026-08-13 01:30:00.037772 UTC,2026-08-13 01:30:00.002400 UTC,25.8,500.0,"""Buy""",6
2026-08-13 08:08:12.739081 UTC,2026-08-13 08:08:12.706680 UTC,25.88,10000.0,"""Buy""",5
…,…,…,…,…,…
2026-08-13 01:32:38.315706 UTC,2026-08-13 01:32:38.281170 UTC,25.86,2000.0,"""Buy""",2
2026-08-13 01:32:42.976887 UTC,2026-08-13 01:32:42.943740 UTC,25.86,88500.0,"""Buy""",2
2026-08-13 06:36:57.197876 UTC,2026-08-13 06:36:57.167550 UTC,25.88,500.0,"""Buy""",2


In [14]:
trades.filter(
    (pl.col("price") <= 0)
    | (pl.col("qty") <= 0)
)

instrument,ingress_ts,transaction_ts,publish_ts,trade_id,price,qty,side,other_data,trade_id_num
str,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,str,str,i64


In [15]:
trades.filter(
    (pl.col("ingress_ts") < pl.col("publish_ts"))
    | (
        pl.col("transaction_ts").is_not_null()
        & (pl.col("publish_ts") < pl.col("transaction_ts"))
    )
)

instrument,ingress_ts,transaction_ts,publish_ts,trade_id,price,qty,side,other_data,trade_id_num
str,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,str,str,i64


In [18]:
trades_latency = trades.with_columns(
    capture_latency_us = (
        pl.col("ingress_ts") - pl.col("publish_ts")
    ).dt.total_microseconds()
)

trades_latency.filter(pl.col("capture_latency_us") > 1_000_000)

trades_latency.select(
    pl.col("capture_latency_us").min().alias("min"),
    pl.col("capture_latency_us").quantile(0.5).alias("median"),
    pl.col("capture_latency_us").quantile(0.95).alias("p95"),
    pl.col("capture_latency_us").quantile(0.99).alias("p99"),
    pl.col("capture_latency_us").max().alias("max"),
)

min,median,p95,p99,max
i64,f64,f64,f64,i64
28048,33294.0,42571.0,49635.0,72333


In [19]:
trade_latency_us = (pl.col("ingress_ts") - pl.col("publish_ts")).dt.total_microseconds()

p95_trade_latency_us = trades.select(trade_latency_us.quantile(0.95)).item()
p99_trade_latency_us = trades.select(trade_latency_us.quantile(0.99)).item()

trades = trades.with_columns(
    capture_latency_us=trade_latency_us,
    latency_gt_p95=trade_latency_us > p95_trade_latency_us,
    latency_gt_p99=trade_latency_us > p99_trade_latency_us,
    latency_gt_1s=trade_latency_us > 1_000_000,
)

In [20]:
trades.null_count()

instrument,ingress_ts,transaction_ts,publish_ts,trade_id,price,qty,side,other_data,trade_id_num,capture_latency_us,latency_gt_p95,latency_gt_p99,latency_gt_1s
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,1513,0,0,0,0,0,9666,0,0,0,0,0


In [21]:
trades.group_by("side").len()

side,len
str,u32
"""Buy""",9666


In [22]:
trades.select(
    pl.col("price").min().alias("min_price"),
    pl.col("price").quantile(0.01).alias("p01_price"),
    pl.col("price").quantile(0.5).alias("median_price"),
    pl.col("price").quantile(0.99).alias("p99_price"),
    pl.col("price").max().alias("max_price"),
    pl.col("qty").min().alias("min_qty"),
    pl.col("qty").quantile(0.01).alias("p01_qty"),
    pl.col("qty").quantile(0.5).alias("median_qty"),
    pl.col("qty").quantile(0.99).alias("p99_qty"),
    pl.col("qty").max().alias("max_qty"),
)

min_price,p01_price,median_price,p99_price,max_price,min_qty,p01_qty,median_qty,p99_qty,max_qty
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
24.0,25.8,25.9,26.0,26.1,1.0,500.0,16000.0,606000.0,1.4595e6
